# Notebook 04: Eval Runner
## Goal: Validate NB01 → NB02 → NB03 against fixtures

Two modes:
- **Mode 1 (End-to-End)**: Raw fixture → CanonicalPacket → NB02 → NB03
- **Mode 2 (Policy-Only)**: CanonicalPacket fixture → NB02 → NB03

In [ ]:
import json
import sys
import os
from dataclasses import asdict
from datetime import datetime
from typing import List, Dict, Any, Optional

# Add paths for fixture imports
sys.path.insert(0, os.path.abspath(os.getcwd()))

# =============================================================================
# SECTION 0: Load notebook models
# =============================================================================

def load_notebook_models(nb_path):
    """Load code from a notebook into a namespace."""
    with open(nb_path, 'r', encoding='utf-8') as f:
        nb = json.load(f)
    ns = {}
    for cell in nb['cells']:
        if cell['cell_type'] == 'code':
            try:
                exec(cell['source'], ns)
            except Exception as e:
                print(f"  Warning loading cell: {e}")
    return ns

# Load NB02 models
nb02_dir = os.path.abspath(os.path.join(os.getcwd()))
nb02 = load_notebook_models(os.path.join(nb02_dir, 'notebooks', '02_gap_and_decision.ipynb'))

# Load NB03 models
nb03 = load_notebook_models(os.path.join(nb02_dir, 'notebooks', '03_session_strategy.ipynb'))

# Pull symbols
CanonicalPacket = nb02['CanonicalPacket']
Slot = nb02['Slot']
EvidenceRef = nb02['EvidenceRef']
UnknownField = nb02['UnknownField']
run_gap_and_decision = nb02['run_gap_and_decision']

SessionStrategy = nb03['SessionStrategy']
PromptBundle = nb03['PromptBundle']
SessionOutput = nb03['SessionOutput']
build_session_output = nb03['build_session_output']
DecisionResult = nb03['DecisionResult']

print("Models loaded.")

In [ ]:
# =============================================================================
# SECTION 1: FIXTURE HELPERS
# =============================================================================

def packet_from_raw_fixture(fixture: Dict[str, Any]) -> CanonicalPacket:
    """
    Convert a raw fixture's expected extracted fields into a CanonicalPacket.
    This simulates what NB01 would produce.
    """
    facts = {}
    expected = fixture["expected"]
    extracted = expected.get("extracted_fields", {})

    for field_name, field_info in extracted.items():
        value = field_info.get("value")
        auth = field_info.get("authority", "explicit_owner")
        conf = field_info.get("confidence", 0.8)
        facts[field_name] = Slot(value=value, confidence=conf, authority_level=auth)

    # Build unknowns
    unknowns = []
    for uf in expected.get("expected_unknowns", []):
        unknowns.append(UnknownField(field_name=uf, reason="not_extracted_yet"))

    # Build contradictions from expected
    contradictions = []
    for ctype in expected.get("expected_contradictions", []):
        # Map contradiction type to field names
        field_map = {
            "date_conflict": "travel_dates",
            "destination_conflict": "destination_city",
            "budget_conflict": "budget_range",
            "traveler_count_conflict": "traveler_count",
            "origin_conflict": "origin_city",
        }
        fname = field_map.get(ctype, ctype)
        # Try to extract values from the facts
        if fname in facts and isinstance(facts[fname].value, list):
            values = facts[fname].value
        else:
            values = ["value_a", "value_b"]
        contradictions.append({
            "field_name": fname,
            "values": values,
            "sources": ["fixture_source"]
        })

    return CanonicalPacket(
        packet_id=fixture["fixture_id"],
        created_at=datetime.now().isoformat(),
        last_updated=datetime.now().isoformat(),
        facts=facts,
        unknowns=unknowns,
        contradictions=contradictions,
        stage="discovery",
    )


def packet_from_packet_fixture(fixture: Dict[str, Any]) -> CanonicalPacket:
    """Return the packet from a packet fixture."""
    return fixture["packet"]

In [ ]:
# =============================================================================
# SECTION 2: EVALUATION ENGINE
# =============================================================================

from dataclasses import dataclass

@dataclass
class EvalResult:
    fixture_id: str
    mode: str  # "e2e" or "policy"
    passed: bool
    checks: List[Dict[str, Any]]
    decision_state_actual: str
    decision_state_expected: str
    errors: List[str]


def evaluate_packet_fixture(fixture: Dict[str, Any]) -> EvalResult:
    """
    Evaluate a CanonicalPacket fixture through NB02 → NB03.
    Checks: decision state, blockers, question quality, internal/external boundary.
    """
    checks = []
    errors = []
    expected = fixture["expected"]

    # Run NB02
    packet = fixture["packet"]
    try:
        decision = run_gap_and_decision(packet)
    except Exception as e:
        return EvalResult(
            fixture_id=fixture["fixture_id"],
            mode="policy",
            passed=False,
            checks=[],
            decision_state_actual="ERROR",
            decision_state_expected=expected.get("decision_state", "unknown"),
            errors=[str(e)],
        )

    expected_state = expected.get("decision_state")

    # Check 1: Decision state correctness
    state_ok = decision.decision_state == expected_state
    checks.append({
        "check": "decision_state",
        "passed": state_ok,
        "expected": expected_state,
        "actual": decision.decision_state,
    })

    # Check 2: Hard blocker correctness
    if "hard_blockers" in expected:
        expected_blockers = set(expected["hard_blockers"])
        actual_blockers = set(decision.hard_blockers)
        blockers_ok = expected_blockers == actual_blockers
        checks.append({
            "check": "hard_blockers",
            "passed": blockers_ok,
            "expected": sorted(expected_blockers),
            "actual": sorted(actual_blockers),
        })

    # Check 3: Soft blocker correctness
    if "soft_blockers" in expected:
        expected_soft = set(expected["soft_blockers"])
        actual_soft = set(decision.soft_blockers)
        soft_ok = expected_soft == actual_soft
        checks.append({
            "check": "soft_blockers",
            "passed": soft_ok,
            "expected": sorted(expected_soft),
            "actual": sorted(actual_soft),
        })

    # Check 4: Question count minimum
    if "question_count_min" in expected:
        min_q = expected["question_count_min"]
        actual_q = len(decision.follow_up_questions)
        q_ok = actual_q >= min_q
        checks.append({
            "check": "question_count_min",
            "passed": q_ok,
            "expected_min": min_q,
            "actual": actual_q,
        })

    # Check 5: Internal vs external boundary
    if expected_state == "PROCEED_INTERNAL_DRAFT":
        internal_ok = "PROCEED_INTERNAL_DRAFT" in str(decision.decision_state)
        checks.append({
            "check": "internal_draft_boundary",
            "passed": internal_ok,
            "expected": "PROCEED_INTERNAL_DRAFT",
            "actual": decision.decision_state,
        })

    if expected_state == "PROCEED_TRAVELER_SAFE":
        safe_ok = decision.decision_state == "PROCEED_TRAVELER_SAFE"
        # Also check that no follow-up questions exist (safe = complete)
        no_questions = len(decision.follow_up_questions) == 0
        checks.append({
            "check": "traveler_safe_complete",
            "passed": safe_ok and no_questions,
            "details": f"state={decision.decision_state}, questions={len(decision.follow_up_questions)}",
        })

    # Check 6: Tone/confidence behavior (for traveler-safe)
    if expected_state == "PROCEED_TRAVELER_SAFE":
        checks.append({
            "check": "confidence_reasonable",
            "passed": decision.confidence_score > 0.3,
            "actual": decision.confidence_score,
        })

    # Check 7: Stop generates review briefing
    if expected_state == "STOP_NEEDS_REVIEW":
        stop_ok = decision.decision_state == "STOP_NEEDS_REVIEW"
        has_questions = len(decision.follow_up_questions) > 0
        checks.append({
            "check": "stop_generates_review",
            "passed": stop_ok,
            "details": f"state={decision.decision_state}, questions={len(decision.follow_up_questions)}",
        })

    passed = all(c["passed"] for c in checks)

    return EvalResult(
        fixture_id=fixture["fixture_id"],
        mode="policy",
        passed=passed,
        checks=checks,
        decision_state_actual=decision.decision_state,
        decision_state_expected=expected_state,
        errors=errors,
    )


def evaluate_raw_fixture(fixture: Dict[str, Any]) -> EvalResult:
    """
    Evaluate a raw fixture through simulated NB01 → NB02 → NB03.
    NB01 is simulated by constructing the packet from expected extracted fields.
    """
    packet = packet_from_raw_fixture(fixture)
    expected = fixture["expected"]

    # Map raw fixture expected fields to policy fixture format
    policy_expected = {
        "decision_state": expected.get("nb02_decision_state"),
        "hard_blockers": expected.get("nb02_hard_blockers", []),
    }
    if "nb03_behavior" in expected:
        policy_expected["nb03_behavior"] = expected["nb03_behavior"]

    # Wrap packet in a fixture for the policy evaluator
    policy_fixture = {
        "fixture_id": fixture["fixture_id"],
        "packet": packet,
        "expected": policy_expected,
    }

    return evaluate_packet_fixture(policy_fixture)

In [ ]:
# =============================================================================
# SECTION 3: RUN EVALUATIONS
# =============================================================================

# Import fixtures
from data.fixtures.raw_fixtures import RAW_FIXTURES
from data.fixtures.packet_fixtures import PACKET_FIXTURES

print(f"Raw fixtures: {len(RAW_FIXTURES)}")
print(f"Packet fixtures: {len(PACKET_FIXTURES)}")
print()

# Run Mode 2: Policy-Only (Packet fixtures)
print("=" * 60)
print("  MODE 2: Policy-Only (CanonicalPacket → NB02 → NB03)")
print("=" * 60)

policy_results = []
for fid, fixture in PACKET_FIXTURES.items():
    result = evaluate_packet_fixture(fixture)
    policy_results.append(result)
    status = "PASS" if result.passed else "FAIL"
    print(f"  [{status}] {fid}: {result.decision_state_actual} (expected: {result.decision_state_expected})")
    if result.errors:
        for e in result.errors:
            print(f"         ERROR: {e}")

# Run Mode 1: End-to-End (Raw fixtures)
print()
print("=" * 60)
print("  MODE 1: End-to-End (Raw → NB01(simulated) → NB02 → NB03)")
print("=" * 60)

e2e_results = []
for fid, fixture in RAW_FIXTURES.items():
    result = evaluate_raw_fixture(fixture)
    e2e_results.append(result)
    status = "PASS" if result.passed else "FAIL"
    print(f"  [{status}] {fid}: {result.decision_state_actual} (expected: {result.decision_state_expected})")
    if result.errors:
        for e in result.errors:
            print(f"         ERROR: {e}")

In [ ]:
# =============================================================================
# SECTION 4: SUMMARY REPORT
# =============================================================================

print()
print("=" * 60)
print("  EVALUATION SUMMARY")
print("=" * 60)

# Mode 2 summary
policy_pass = sum(1 for r in policy_results if r.passed)
policy_fail = sum(1 for r in policy_results if not r.passed)
print(f"\n  Mode 2 (Policy-Only): {policy_pass}/{len(policy_results)} passed")
for r in policy_results:
    if not r.passed:
        print(f"    ✗ {r.fixture_id}: expected {r.decision_state_expected}, got {r.decision_state_actual}")
        for c in r.checks:
            if not c["passed"]:
                print(f"      Failed check: {c['check']}")
                if "expected" in c:
                    print(f"        Expected: {c['expected']}")
                if "actual" in c:
                    print(f"        Actual: {c['actual']}")

# Mode 1 summary
e2e_pass = sum(1 for r in e2e_results if r.passed)
e2e_fail = sum(1 for r in e2e_results if not r.passed)
print(f"\n  Mode 1 (End-to-End): {e2e_pass}/{len(e2e_results)} passed")
for r in e2e_results:
    if not r.passed:
        print(f"    ✗ {r.fixture_id}: expected {r.decision_state_expected}, got {r.decision_state_actual}")
        for c in r.checks:
            if not c["passed"]:
                print(f"      Failed check: {c['check']}")
                if "expected" in c:
                    print(f"        Expected: {c['expected']}")
                if "actual" in c:
                    print(f"        Actual: {c['actual']}")

# Overall
all_pass = policy_pass + e2e_pass
all_total = len(policy_results) + len(e2e_results)
print(f"\n  Overall: {all_pass}/{all_total} passed ({all_pass/all_total*100:.0f}%)")

# Save results as JSON
all_results = []
for r in policy_results + e2e_results:
    all_results.append({
        "fixture_id": r.fixture_id,
        "mode": r.mode,
        "passed": r.passed,
        "decision_state_actual": r.decision_state_actual,
        "decision_state_expected": r.decision_state_expected,
        "checks": r.checks,
        "errors": r.errors,
    })

with open(os.path.join(nb02_dir, 'data', 'fixtures', 'eval_results.json'), 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
print(f"\n  Results saved to data/fixtures/eval_results.json")